# 05. Deep Agents 입문 — 올인원 에이전트

Deep Agents SDK의 `create_deep_agent()`로 도구·메모리·백엔드가 내장된 에이전트를 한 줄로 만들어 봅니다.

## 학습 목표

- `create_deep_agent()`로 에이전트를 생성한다
- `invoke()`로 에이전트를 실행한다
- 커스텀 도구를 추가한 에이전트를 만든다

## 5.1 환경 설정

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-4.1")
print("\u2713 모델 준비 완료")

✓ 모델 준비 완료


In [2]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


Langfuse tracing ON — https://lf.ddok.ai


## 5.2 에이전트 생성 

`create_deep_agent()`는 LangChain 모델을 받아, 파일 읽기/쓰기/검색 등의 **빌트인 도구**가 자동으로 포함된 에이전트를 반환합니다.
반환 타입이 LangGraph의 `CompiledStateGraph`이므로 `invoke()`, `stream()` 등을 그대로 쓸 수 있습니다.

**Deep Agents란?**

Deep Agents는 에이전트 개발을 단순하게 만들기 위해 설계된 프레임워크로, **에이전트 하네스(harness)** 역할을 합니다. LangChain의 기본 에이전트 컴포넌트 위에 구축되며, 실행 관리에는 LangGraph를 씁니다.

**핵심 내장 기능:**

| 기능 | 설명 |
|---|---|
| **태스크 플래닝** | `write_todos` 도구로 복잡한 문제를 관리 가능한 단계로 자동 분해 |
| **컨텍스트 관리** | 파일 시스템 도구(`write_file`, `read_file`)로 토큰 한도를 넘지 않고 대량 데이터 처리 |
| **유연한 저장소** | 인메모리, 로컬 디스크, 영구 저장소, 샌드박스 환경 등 플러거블 백엔드 지원 |
| **서브에이전트 위임** | 특정 하위 작업을 위한 전문 서브에이전트를 만들어 컨텍스트 격리 |
| **영구 메모리** | LangGraph 메모리 인프라로 여러 대화에 걸쳐 정보를 유지 |

**에이전트 생성 방법:**

`create_deep_agent()`에 모델, 도구, 시스템 프롬프트를 전달하면 됩니다. 도구 호출을 지원하는 모델이 필요하며, Anthropic, OpenAI 등 다양한 프로바이더를 쓸 수 있습니다.

In [3]:
from deepagents import create_deep_agent

agent = create_deep_agent(model=model)
print(f"\u2713 에이전트 생성 완료 (타입: {type(agent).__name__})")

✓ 에이전트 생성 완료 (타입: CompiledStateGraph)


In [4]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "안녕하세요! 어떤 도구를 사용할 수 있나요? 목록만 간단히 알려주세요."
            }
        ]
    },
    config=lf_config,
)

print(result["messages"][-1].content)

사용 가능한 도구 목록입니다:

1. 파일시스템 도구: ls, read_file, write_file, edit_file, glob, grep
2. 할 일 관리: write_todos
3. 태스크(서브에이전트) 실행: task
4. 병렬 도구 실행: multi_tool_use.parallel

필요한 도구의 기능 등, 더 자세한 설명이 필요하시면 말씀해 주세요!


## 5.3 커스텀 도구 추가

Python 함수에 **docstring**과 **타입 힌트**를 작성하면 그대로 도구가 됩니다.

**커스텀 도구의 동작 원리:**

커스텀 도구는 일반 Python 함수로 작성하며, 다음 두 가지가 자동으로 변환됩니다:

- **docstring** → 도구 설명 (에이전트가 이 도구를 언제 써야 하는지 판단하는 근거)
- **타입 힌트** → 파라미터 스키마 (에이전트가 올바른 인자를 넘기는 데 사용)

`create_deep_agent()`의 `tools` 파라미터에 함수 리스트를 전달하면, 빌트인 도구(파일 읽기/쓰기, todo 등)와 함께 에이전트 도구 목록에 추가됩니다. `system_prompt` 파라미터로 에이전트의 행동 방식을 지정할 수도 있습니다.

In [5]:
def greet(name: str) -> str:
    """이름으로 인사합니다."""
    return f"안녕하세요, {name}님! Deep Agents에 오신 것을 환영합니다."

custom_agent = create_deep_agent(
    model=model,
    tools=[greet],
    system_prompt="당신은 친절한 어시스턴트입니다. 누군가 자기소개를 하면 greet 도구를 사용하세요.",
)

result = custom_agent.invoke(
    {"messages": [{"role": "user", "content": "제 이름은 앨리스입니다."}]},
    config=lf_config,
)
print(result["messages"][-1].content)

안녕하세요, 앨리스님! Deep Agents에 오신 것을 환영합니다. 무엇을 도와드릴까요?


## 요약

| 핵심 API | 역할 |
|---|---|
| `create_deep_agent(model)` | 빌트인 도구가 포함된 에이전트 생성 |
| `create_deep_agent(model, tools, system_prompt)` | 커스텀 도구 + 시스템 프롬프트 |
| `agent.invoke()` | 에이전트 실행 |

### 다음 단계
→ **[06_comparison.ipynb](./06_comparison.ipynb)**: 세 프레임워크를 비교하고 중급 과정을 안내합니다.